In [ ]:
import pandas as pd
import math
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.metrics import precision_at_k, recall_at_k, ndcg_at_k, evaluate, safe_mean
from src.baseline import popular_movies, recommend_popular

In [56]:
ratings = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    names=["user_id", "movie_id", "rating", "timestamp"],
    engine="python",
)
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [57]:
ratings.shape

(1000209, 4)

In [58]:
ratings["user_id"].nunique()

6040

In [59]:
n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
n_ratings = len(ratings)

density = n_ratings / (n_users * n_movies)
print(density)

0.044683625622312845


In [60]:
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
print(ratings["datetime"].min())
print(ratings["datetime"].max())

2000-04-25 23:05:32
2003-02-28 17:49:50


In [61]:
ratings_sorted = ratings.sort_values("timestamp").reset_index(drop=True)
cutoff = int(len(ratings_sorted) * 0.8)
train = ratings_sorted[:cutoff]
test = ratings_sorted[cutoff:]
print(len(train))
print(len(test))
print(len(ratings_sorted))

800167
200042
1000209


In [62]:
train_users = set(train["user_id"])
test_users = set(test["user_id"])
print(len(test_users))
print(len(test_users - train_users))
print(len(test_users - train_users) / len(test_users)) 

1783
640
0.35894559730790804


In [63]:
test_warm = test[test["user_id"].isin(train_users)]
test["user_id"].nunique() - test_warm["user_id"].nunique()

640

In [64]:
ndcg_at_k([20, 10, 30], {20}, 3)

1.0

In [65]:
warm_users = list(test_warm["user_id"].unique())
recs = recommend_popular(train, warm_users, k=10)

relevant_ratings = test_warm[test_warm["rating"] >= 4]
relevant_by_user = relevant_ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

results = evaluate(recs, relevant_by_user, k=10)
print(results)

{'precision': 0.19588801399825023, 'recall': 0.04925529934025789, 'ndcg': 0.21449499883811188}
